# Nyaya — an open Indian legal guidance system (start here)

Ask a legal question in **English, Hindi or Hinglish** and get the sections of **current Indian law**
it resolves to — the Bharatiya Nyaya Sanhita, Bharatiya Nagarik Suraksha Sanhita and Bharatiya Sakshya
Adhiniyam in force since July 2024, plus 24 other central acts and the Constitution — and, in the full
system, a plain-language answer cited to those sections.

This notebook is the landing page for the Nyaya work on Kaggle. It runs on CPU in about three minutes:
it installs the open-source package, downloads the public statute database, and runs the retriever on
three questions. The GPU notebooks that produced every published number are linked at the end.

> **⚖️ Not legal advice.** Nyaya provides legal *information*. The practice of law in India is reserved
> to advocates enrolled under the Advocates Act, 1961. Free legal aid: NALSA / DLSA.

**Where things live**
- Code, evaluation harness, every prediction behind every number: [github.com/JitendraJha98/nyaya-model](https://github.com/JitendraJha98/nyaya-model) (Apache-2.0, release v0.3.0)
- Models and data: [huggingface.co/NyayaLabs98](https://huggingface.co/NyayaLabs98) — statute DB, `nyaya-embed-v1`, `nyaya-reranker-mini-v1`, training data
- Browser demo, nothing to install: [huggingface.co/spaces/NyayaLabs98/nyaya-demo](https://huggingface.co/spaces/NyayaLabs98/nyaya-demo)


In [ ]:
# Install the open-source package from the tagged release (standard-library retriever; no GPU needed)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "git+https://github.com/JitendraJha98/nyaya-model@v0.3.0"], check=True)
import nyaya
print("nyaya", nyaya.__version__)


In [ ]:
# Three questions, three scripts. The statute DB (about 5 MB) downloads once from the public Hub dataset.
import subprocess, sys

QUESTIONS = [
    "police FIR nahi likh rahi, kya karu?",                              # Hinglish
    "What is the punishment for cheque bounce?",                         # English
    "पति के परिवार द्वारा दहेज की मांग पर क्या कानूनी कार्रवाई हो सकती है?",  # Hindi
]
for q in QUESTIONS:
    print("=" * 100)
    print("Q:", q)
    print("-" * 100)
    out = subprocess.run([sys.executable, "-m", "nyaya.cli", "ask", q, "--k", "3"], capture_output=True, text=True)
    text = out.stdout or out.stderr
    # print the section headings and the first line of each section; the full text is in the CLI output
    for line in text.splitlines():
        if line.startswith("Section ") or line.startswith("Article ") or "official guidance" in line or line.startswith("⚖"):
            print(line[:160])


## How it works, in one paragraph

A question goes to a retriever over the statute database: exact-citation lookup (old IPC/CrPC numbers are
bridged through the official mapping tables), BM25 with a lay-to-statute vocabulary, and a dense stage
(`nyaya-embed-v1`, fine-tuned from multilingual-e5-base on the project's own pairs) fused by reciprocal
rank. A calibrated coverage gate says when no indexed act can answer. The top sections are handed to a
small open reader — `Qwen/Qwen3-4B-Instruct-2507` by default — which writes a plain-language answer citing
only what it was given. A scorer grades every answer against gold facts and citations, and every
prediction is kept so anyone can re-score it on a CPU.

What it does not do: case law, state law, Hindi statute text (India Code's Hindi PDFs are image scans),
or legal advice.


## What the measurements say

Nyaya-Eval-v1, 413 citizen questions, 768 new tokens, k=8 retrieved sections, one Kaggle T4. Every row is a
paired comparison on identical questions with a 10,000-round bootstrap; predictions are committed in the
repository under `outputs/eval-v1/`.

| configuration | fact recall | citation accuracy | vs. first release |
|---|---|---|---|
| Qwen2.5-3B + BM25 + zero-shot e5-base (first release) | 35.8% | 55.6% | — |
| Qwen2.5-3B + BM25 + **nyaya-embed-v1** | 39.7% | 61.1% | +3.9, CI [+0.9, +7.0] |
| **Qwen3-4B-Instruct-2507** + BM25 + e5-base | 50.6% | 72.2% | +14.8, CI [+11.4, +18.3] |
| **Qwen3-4B + nyaya-embed-v1** (default) | **52.0%** | **77.1%** | **+16.2, CI [+12.7, +19.9]** |

Five fine-tunes of the 3B model never beat their base model, and the repository says so. What moved the
score was a retriever trained on the project's own question–section pairs and a newer-generation open
reader. External check on BhashaBench-Legal (3,000 paired questions, no retrieval): Qwen3-4B 52.5% vs
49.6% for the 3B base, most of the gain on Hindi.


## The research notebooks (GPU T4, all on this account)

| notebook | what it measured |
|---|---|
| `nyaya-shootout` | Six small readers under one fixed retriever; Qwen3-4B wins by 14.8 points, Llama-3.2-3B ties the 3B base, Gemma-3-4B cannot run in fp16 on a T4 |
| `nyaya-train-retriever` | Trains `nyaya-embed-v1` and `nyaya-reranker-mini-v1` on 4,712 question–section pairs (2 + 4.5 minutes of T4) |
| `nyaya-retriever-effect` | Same reader, retriever swapped: +3.9 points, the project's first gain with an interval clear of zero |
| `nyaya-qwen3-embed` | The default configuration, paired against every parent |
| `nyaya-bhashabench` | External MCQ benchmark, both readers on the same 3,000 questions, option-letter logit scoring |
| `nyaya-teacher` | A served 14B teacher passes its gate against the 3B and still scores below the 4B, so no distillation |

Every notebook keeps its own `progress.txt` and copies its outputs to the Output tab; the repository's
`docs/RESULTS.md` is the full write-up, including the measurement mistakes that were found and corrected.
